# Практикум 17. Стресс-тест алгоритмов NLP на агглютинативной морфологии

В лаборатории 17 вы видели, как быстро растёт число узбекских словоформ и как простые алгоритмы ломаются на апострофах, кириллице и чередованиях. Здесь повторим всё на коде:

1. напишем генератор форм узбекского существительного и глагола с правильным порядком суффиксов (морфотактикой) и алломорфией;
2. по закону Хипса и кривым OOV увидим, почему словарь агглютинативного языка почти не насыщается;
3. обучим BPE с нуля и сравним его с морфемной сегментацией: фертильность и совпадение границ;
4. прогоним набор искажений (апострофы, кириллица, цепочки суффиксов, k→g, выпадение гласного) через наивный и устойчивый конвейер и получим таблицу «прошёл / не прошёл».

Материал узбекский, к корням и суффиксам даны русские переводы. Где полезно, сравниваем с русским (библиотека `pymorphy3`).

*Практикум к лаборатории 17 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

In [ ]:
!pip install -q pymorphy3

## 1. Взрыв форм: генератор узбекских словоформ

Узбекское существительное присоединяет суффиксы в строгом порядке (морфотактика): корень → число (-lar) → принадлежность (-im «мой», -ing «твой», -i «его»…) → падеж (-ning, -ni, -ga, -da, -dan). После местного падежа -da может идти суффикс -gi («находящийся в…»), а за ним снова число и падеж: kitob·lar·imiz·da·gi·lar·dan «из тех, что в наших книгах». В конце может стоять вопросительная частица -mi.

Форма суффикса зависит от соседей (алломорфия):
- после гласного bola·m «мой ребёнок», bola·si, после согласного kitob·im «моя книга», kitob·i;
- -ga после k и q превращается в -ka и -qa: terak·ka «к тополю», qishloq·qa «в село»;
- перед гласным суффиксом принадлежности конечные k, q многих многосложных корней озвончаются: terak → terag·im, qishloq → qishlogʻ·imiz;
- в некоторых корнях выпадает гласный последнего слога: ogʻiz «рот» → ogʻz·i «его рот», burun «нос» → burn·i, shahar «город» → shahr·i.

Генератор ниже выполняет эти правила (-dagi он делит на две морфемы da·gi; в лаборатории -dagi был одним вариантом падежной позиции). В скобках показано, что получилось бы при простом склеивании суффиксов без правил.

In [ ]:
import re, itertools, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

VOWELS = ("a", "e", "i", "o", "u", "oʻ")

# корень -> что происходит с основой перед гласным суффиксом принадлежности:
# "kq": конечный k -> g, q -> gʻ (terak -> teragi); "drop": выпадает гласный последнего слога (ogʻiz -> ogʻzi)
NOUNS = {
    "kitob": "", "uy": "", "bola": "", "maktab": "", "dala": "", "olma": "", "bogʻ": "", "togʻ": "",
    "qoʻl": "", "doʻst": "", "oʻqituvchi": "", "maʼno": "", "sheʼr": "", "talaba": "", "kutubxona": "",
    "qishloq": "kq", "terak": "kq", "yurak": "kq", "quloq": "kq", "eshik": "kq",
    "ogʻiz": "drop", "burun": "drop", "shahar": "drop", "oʻgʻil": "drop", "qorin": "drop",
}
GLOSS = {
    "kitob": "книга", "uy": "дом", "bola": "ребёнок", "maktab": "школа", "dala": "поле", "olma": "яблоко",
    "bogʻ": "сад", "togʻ": "гора", "qoʻl": "рука", "doʻst": "друг", "oʻqituvchi": "учитель", "maʼno": "смысл",
    "sheʼr": "стихотворение", "talaba": "студент", "kutubxona": "библиотека", "qishloq": "село", "terak": "тополь", "yurak": "сердце", "quloq": "ухо",
    "eshik": "дверь", "ogʻiz": "рот", "burun": "нос", "shahar": "город", "oʻgʻil": "сын", "qorin": "живот",
    "yoz": "писать", "oʻqi": "читать", "ishla": "работать", "kel": "приходить", "chiq": "выходить",
}
TAGS = {
    "PL": "мн.ч.", "P1": "мой", "P2": "твой", "P3": "его/её", "P1PL": "наш", "P2PL": "ваш",
    "GEN": "род.п.", "ACC": "вин.п.", "DAT": "дат.п.", "LOC": "мест.п.", "ABL": "исх.п.",
    "ATTR": "«находящийся в»", "Q": "вопрос",
    "NEG": "отриц.", "PAST": "прош. -di", "PRES": "наст.-буд. -a/-y", "PROG": "наст. длит. -yap", "PERF": "прош. -gan",
    "1SG": "1 л. ед.", "2SG": "2 л. ед.", "3SG": "3 л.", "1PL": "1 л. мн.", "2PL": "2 л. мн.", "3PL": "3 л. мн.",
}

POSS = {"P1": ("im", "m"), "P2": ("ing", "ng"), "P3": ("i", "si"), "P1PL": ("imiz", "miz"), "P2PL": ("ingiz", "ngiz")}
CASE = {"GEN": "ning", "ACC": "ni", "DAT": "ga", "LOC": "da", "ABL": "dan"}

def word(m):
    return "".join(x for x, _ in m)

def changed_stem(root):
    kind = NOUNS.get(root, "")
    if kind == "kq":                      # terak -> terag, qishloq -> qishlogʻ
        return root[:-1] + ("g" if root.endswith("k") else "gʻ")
    if kind == "drop":                    # ogʻiz -> ogʻz, shahar -> shahr
        return root[:-2] + root[-1]
    return root

def noun_form(root, pl=False, poss=None, case=None, dagi=False, pl2=False, case2=None, q=False, rules=True):
    """Форма существительного: список пар (морф, тег). rules=False склеивает суффиксы без правил алломорфии."""
    m = [[root, "ROOT"]]
    def add_case(c):
        suf = CASE[c]
        if rules and c == "DAT" and word(m).endswith("k"):
            suf = "ka"                    # terak + ga -> terakka
        elif rules and c == "DAT" and word(m).endswith("q"):
            suf = "qa"                    # qishloq + ga -> qishloqqa
        m.append([suf, c])
    if pl:
        m.append(["lar", "PL"])
    if poss:
        after_cons, after_vowel = POSS[poss]
        if rules and word(m).endswith(VOWELS):
            m.append([after_vowel, poss])  # bola + m
        else:
            if rules and not pl:
                m[0][0] = changed_stem(root)
            m.append([after_cons, poss])   # kitob + im
    if case:
        add_case(case)
    if dagi:                               # -gi идёт только после местного падежа -da
        m.append(["gi", "ATTR"])
        if pl2:
            m.append(["lar", "PL"])
        if case2:
            add_case(case2)
    if q:
        m.append(["mi", "Q"])
    return [tuple(x) for x in m]

def show(root, **kw):
    m = noun_form(root, **kw)
    plain = word(noun_form(root, rules=False, **kw))
    gloss = " + ".join([GLOSS.get(root, root)] + [TAGS[t] for _, t in m[1:]])
    note = "" if plain == word(m) else "   (без правил: " + plain + ")"
    print(f"{'·'.join(x for x, _ in m):<34} {gloss}{note}")

show("kitob", pl=True, poss="P1PL", case="LOC", dagi=True, pl2=True, case2="ABL")
show("bola", poss="P1")
show("bola", pl=True, poss="P3", case="DAT", q=True)
show("terak", case="DAT")
show("terak", poss="P1", case="GEN")
show("qishloq", case="DAT")
show("qishloq", poss="P1PL", case="DAT")
show("qishloq", pl=True, poss="P1PL", case="DAT")
show("ogʻiz", poss="P3")
show("burun", poss="P3", case="ABL")
show("shahar", poss="P1PL", case="LOC", dagi=True)
show("shahar", pl=True, poss="P1PL", case="LOC")

Сколько форм у одного корня? Число (2) × принадлежность (6, считая «нет») × падеж (6) = 72, как в конструкторе лаборатории 1; затем цепочка -dagi и частица -mi. У глагола: утвердительная/отрицательная форма × 4 времени × 6 лиц × -mi. Это лишь малая часть настоящей парадигмы: нет залогов, наклонений, деепричастий, причастий и многого другого.

Сравните с русским: в «книг-ами» одно окончание выражает сразу число и падеж (фузия), поэтому у существительного около десятка форм. В узбекском каждое значение получает свой суффикс, и числа форм перемножаются. В конце ячейки для сравнения посчитаем русские парадигмы с `pymorphy3` (у «книги» он покажет 10 строк: к девяти из лаборатории добавится «книгою», старый вариант творительного падежа).

In [ ]:
POSS_OPTS = [None] + list(POSS)
CASE_OPTS = [None] + list(CASE)
CASE2_OPTS = [None, "ACC", "DAT", "ABL"]      # падеж после -dagi(-lar)

def noun_paradigm(root):
    forms = []
    for pl, poss, case in itertools.product([False, True], POSS_OPTS, CASE_OPTS):
        chains = [{}]
        if case == "LOC":
            chains += [dict(dagi=True, pl2=p2, case2=c2) for p2 in (False, True) for c2 in CASE2_OPTS]
        for ch in chains:
            for q in (False, True):
                forms.append(noun_form(root, pl=pl, poss=poss, case=case, q=q, **ch))
    return forms

par = noun_paradigm("kitob")
steps = [("корень", 1), ("+ -lar", 2), ("+ принадлежность", 2 * 6), ("+ падеж", 2 * 6 * 6),
         ("+ -dagi (-lar) (+ падеж)", len(par) // 2), ("+ -mi", len(par))]
for name, n in steps:
    print(f"{name:<32} {n:4d}")
print(f"Форм у существительного kitob: {len(par)} форм, {len({word(m) for m in par})} разных\n")

# глагол: корень (+ отрицание -ma) + время + лицо (+ -mi)
VERBS = ["yoz", "oʻqi", "ishla", "kel", "chiq"]
TENSE = {"PAST": "di", "PRES": "a", "PROG": "yap", "PERF": "gan"}
PERSON = {
    "PAST": ["m", "ng", "", "k", "ngiz", "lar"],
    "PRES": ["man", "san", "di", "miz", "siz", "dilar"],
    "PROG": ["man", "san", "ti", "miz", "siz", "tilar"],
    "PERF": ["man", "san", "", "miz", "siz", "lar"],
}
PERSON_TAGS = ["1SG", "2SG", "3SG", "1PL", "2PL", "3PL"]

def verb_form(root, tense, person, neg=False, q=False):
    m = [(root, "ROOT")]
    if neg:
        m.append(("ma", "NEG"))
    suf = TENSE[tense]
    if tense == "PRES" and word(m).endswith(VOWELS):
        suf = "y"                          # oʻqi-y-man, yoz-ma-y-man
    elif tense == "PERF" and word(m).endswith("k"):
        suf = "kan"                        # ek-kan
    elif tense == "PERF" and word(m).endswith("q"):
        suf = "qan"                        # chiq-qan
    m.append((suf, tense))
    if PERSON[tense][person]:
        m.append((PERSON[tense][person], PERSON_TAGS[person]))
    if q:
        m.append(("mi", "Q"))
    return m

def verb_paradigm(root):
    return [verb_form(root, t, p, neg, q) for t in TENSE for p in range(6) for neg in (False, True) for q in (False, True)]

for v in ["yoz", "oʻqi", "chiq"]:
    table = pd.DataFrame([[word(verb_form(v, t, p)) for p in range(6)] for t in TENSE],
                         index=[TAGS[t] for t in TENSE], columns=[TAGS[p] for p in PERSON_TAGS])
    print(v + (" — " + GLOSS[v] if v in GLOSS else ""))
    print(table.to_string(), "\n")
print(f"Форм у глагола yoz: {len(verb_paradigm('yoz'))} форм, например:", word(verb_form("yoz", "PRES", 0, neg=True, q=True)))

# инвентарь: словоформа -> (корень, морфемы); пригодится в разделах 3 и 4
INVENTORY, total = {}, 0
for r in NOUNS:
    for m in noun_paradigm(r):
        INVENTORY[word(m)] = (r, m); total += 1
for v in VERBS:
    for m in verb_paradigm(v):
        INVENTORY[word(m)] = (v, m); total += 1
print(f"Инвентарь: {len(INVENTORY)} словоформ от {len(NOUNS) + len(VERBS)} корней; совпадений между корнями: {total - len(INVENTORY)}")

plt.figure(figsize=(7, 3))
plt.bar([s for s, _ in steps], [n for _, n in steps], color="#2B55C9")
for i, (_, n) in enumerate(steps):
    plt.text(i, n, str(n), ha="center", va="bottom")
plt.xticks(rotation=20, ha="right")
plt.ylabel("число форм")
plt.title("Рост числа форм kitob по слотам")
plt.tight_layout()
plt.show()


import pymorphy3
morph = pymorphy3.MorphAnalyzer()

book = {f.word for f in morph.parse("книга")[0].lexeme}
print(f"книга: {len(book)} разных форм:", ", ".join(sorted(book)))
write = morph.parse("писать")[0].lexeme
finite = {f.word for f in write if f.tag.POS in ("VERB", "INFN")}
print(f"писать: {len(finite)} форм (инфинитив, настоящее, прошедшее, повелительное):", ", ".join(sorted(finite)))
print(f"   вместе с причастиями и деепричастиями: {len({f.word for f in write})}")

## 2. Закон Хипса и OOV: словарь, который не насыщается

Закон Хипса: число разных словоформ $V$ в тексте из $N$ токенов растёт примерно как $V = K N^{\beta}$, $0 < \beta < 1$. Чем больше у языка форм на одну лемму, тем больше $\beta$: новые формы появляются даже тогда, когда новые леммы почти кончились.

Сравним три смоделированных корпуса с одинаковым распределением лемм (закон Ципфа, 4000 лемм, как в лаборатории): «английский» (4 формы на лемму), «русский» (12) и «узбекский» (336, как у нашего генератора; в лаборатории было 168, там после -dagi суффиксы не продолжались). Формы внутри леммы тоже распределены по Ципфу: несколько форм частые, остальные редкие. $\beta$ найдём двумя способами: прямой в log-log (`np.polyfit`) и подгонкой степенной функции в обычной шкале (`scipy.optimize.curve_fit`).

In [ ]:
from scipy.optimize import curve_fit

rng = np.random.default_rng(17)
N_LEMMAS, N_TOKENS, N_TEST = 4000, 200_000, 20_000
FORMS_PER_LEMMA = {"английский": 4, "русский": 12, "узбекский": len(par)}

def zipf(n, s=1.0):
    p = 1.0 / np.arange(1, n + 1) ** s
    return p / p.sum()

corpora = {}
for name, n_forms in FORMS_PER_LEMMA.items():
    lemmas = rng.choice(N_LEMMAS, size=N_TOKENS, p=zipf(N_LEMMAS))
    forms = rng.choice(n_forms, size=N_TOKENS, p=zipf(n_forms))
    corpora[name] = (lemmas, lemmas * 1000 + forms)     # код словоформы = лемма * 1000 + номер формы

def vocab_growth(tokens):
    first = np.zeros(len(tokens), dtype=bool)
    first[np.unique(tokens, return_index=True)[1]] = True   # где слово встретилось впервые
    return np.cumsum(first)                                  # V(N) для каждого N

Ns = np.unique(np.logspace(2, np.log10(N_TOKENS - N_TEST), 30).astype(int))
plt.figure(figsize=(6, 4))
for name, (lem, tok) in corpora.items():
    V = vocab_growth(tok[:N_TOKENS - N_TEST])[Ns - 1]
    beta, logK = np.polyfit(np.log(Ns), np.log(V), 1)        # прямая в log-log: log V = log K + β log N
    (K2, beta2), _ = curve_fit(lambda n, K, b: K * n ** b, Ns, V, p0=(np.exp(logK), beta))
    print(f"{name:<11} V({Ns[-1]}) = {V[-1]:6d}   polyfit (log-log): β = {beta:.3f}, K = {np.exp(logK):5.2f}   curve_fit: β = {beta2:.3f}, K = {K2:5.2f}")
    plt.loglog(Ns, V, "o-", ms=3, label=f"{name}: β = {beta:.2f}")
plt.xlabel("токенов в корпусе N")
plt.ylabel("разных словоформ V(N)")
plt.title("Закон Хипса")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

Порядок языков в обеих подгонках одинаков, а сами $\beta$ различаются: в log-log все точки равноправны, а в обычной шкале подгонку определяют самые большие $N$. Закон Хипса — приближение, и $\beta$ зависит от способа оценки, поэтому сравнивать языки нужно одним и тем же способом.

**OOV** (out of vocabulary) — доля токенов тестового текста, которых не было в обучающем корпусе. Посчитаем её для разных размеров обучающего корпуса дважды: по словоформам и по леммам (как будто у нас есть безошибочный лемматизатор). Сравните сплошные и пунктирные линии: по леммам три языка почти не различаются, вся разница создаётся морфологией.

In [ ]:
train_sizes = [1000, 2000, 5000, 10_000, 20_000, 50_000, 100_000, 180_000]
rows = {}
plt.figure(figsize=(6, 4))
for (name, (lem, tok)), color in zip(corpora.items(), ["#17907E", "#2B55C9", "#C94B2B"]):
    test_tok, test_lem = tok[-N_TEST:], lem[-N_TEST:]
    oov_forms = [np.mean(~np.isin(test_tok, tok[:n])) for n in train_sizes]
    oov_lemmas = [np.mean(~np.isin(test_lem, lem[:n])) for n in train_sizes]
    rows[(name, "формы")] = oov_forms
    rows[(name, "леммы")] = oov_lemmas
    plt.semilogx(train_sizes, oov_forms, "o-", color=color, label=name)
    plt.semilogx(train_sizes, oov_lemmas, "--", color=color)
print("OOV на тесте, %")
print((pd.DataFrame(rows, index=train_sizes).T * 100).round(1).to_string())
plt.xlabel("обучающих токенов")
plt.ylabel("OOV")
plt.title("Доля OOV: сплошные — формы, пунктир — леммы")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 3. BPE против морфем: фертильность и границы

BPE (тема 2) раз за разом склеивает самую частую пару соседних символов. Обучим его с нуля на 45 узбекских предложениях (30 из лаборатории и 15 с чередованиями) и разрежем длинные формы. Две меры качества:
- **фертильность** (fertility) — среднее число токенов на слово (у токенизации по пробелам 1, у посимвольной — длина слова);
- **совпадение границ**: какая доля границ BPE совпадает с границами морфем (точность) и какая доля морфемных границ найдена (полнота), $F_1 = 2PR/(P+R)$.

Правильная сегментация берётся из генератора раздела 1: он знает, где кончается каждая морфема.

In [ ]:
from collections import Counter

UZ_CORPUS = [
    # 30 предложений из лаборатории 17
    "kitoblar stol ustida turibdi", "bu kitobni kutubxonadan oldim", "kitobimiz juda qiziqarli ekan",
    "talabalar kutubxonaga bordilar", "biz uyimizda kitob oʻqiymiz", "uylarimiz maktabga yaqin",
    "maktabimizda katta kutubxona bor", "oʻqituvchimiz talabalarga kitob berdi", "talabalarimiz imtihondan oʻtdilar",
    "doʻstlarim shahardan keldilar", "shahrimizdagi maktablar yangi", "kutubxonadagi kitoblarni oʻqidingizmi",
    "bolalar hovlida oʻynayaptilar", "bolalarimiz maktabdan qaytdilar", "men xat yozdim", "u kitob yozyapti",
    "biz maktabda ishlaymiz", "oʻqituvchilar yigʻilishda gapirdilar", "qishlogʻimizdagi uylar chiroyli",
    "doʻstimning kitobini oʻqidim", "ular kutubxonada ishlayaptilar", "kitoblaringizni stolga qoʻying",
    "talabaning daftari qayerda", "maktablarda imtihonlar boshlandi", "bolalarning daftarlari stolda",
    "uyingizdan maktabgacha uzoqmi", "doʻstlarimizga xat yozdik", "kutubxonamizdagi kitoblar eski",
    "talabalar uylariga qaytdilar", "oʻqituvchimizning kitobi yangi",
    # ещё 15 предложений с чередованиями в основе
    "Qishlogʻimizda katta bogʻ bor.", "Bogʻimizdagi teraklar juda baland.", "Oʻgʻlim maktabdan kech qaytdi.",
    "Dehqonlar daladan qaytishdi.", "Biz qishloqqa yozda boramiz.", "Shahrimizda koʻp maktablar bor.",
    "Uyimizning derazasidan togʻlar koʻrinadi.", "Teragimizning barglari sargʻaydi.", "Bolaning ogʻzi ochiq qoldi.",
    "Sovuqdan uning burni qizarib ketdi.", "Bolalarimizga yangi daftarlar oldik.", "Shaharlarimiz tobora chiroyli boʻlmoqda.",
    "Qishloqlarimizdagi maktablar yangilandi.", "Dalalarimizdan paxta terildi.", "Qoʻlimdagi kitobni doʻstimga berdim.",
]

def letters(w):
    return re.findall(r"[og]ʻ|.", w)          # oʻ и gʻ — одна буква

def words_of(s):
    return re.findall(r"[a-zʻʼ]+", s.lower())

word_counts = Counter(w for s in UZ_CORPUS for w in words_of(s))
print(f"{len(UZ_CORPUS)} предложений, {sum(word_counts.values())} словоупотреблений, {len(word_counts)} разных слов\n")

def merge_pair(sym, a, b):
    out, i = [], 0
    while i < len(sym):
        if i + 1 < len(sym) and sym[i] == a and sym[i + 1] == b:
            out.append(a + b); i += 2
        else:
            out.append(sym[i]); i += 1
    return out

def train_bpe(word_counts, max_merges=1000, verbose=12):
    vocab = {tuple(letters(w)) + ("</w>",): c for w, c in word_counts.items()}
    merges = []
    for step in range(max_merges):
        pairs = Counter()
        for sym, c in vocab.items():
            for a, b in zip(sym, sym[1:]):
                pairs[a, b] += c
        if not pairs:
            break
        (a, b), freq = max(pairs.items(), key=lambda t: (t[1], t[0]))   # самая частая пара
        if freq < 2:                                                     # пары, встреченные один раз, не склеиваем
            break
        merges.append((a, b))
        vocab = {tuple(merge_pair(list(sym), a, b)): c for sym, c in vocab.items()}
        if step < verbose:
            print(f"слияние {step + 1:2d}: {a} + {b} -> {a + b}   ({freq})")
    return merges

def apply_bpe(w, merges):
    sym = letters(w) + ["</w>"]
    for a, b in merges:
        sym = merge_pair(sym, a, b)
    sym[-1] = sym[-1].replace("</w>", "")
    return [s for s in sym if s]

merges = train_bpe(word_counts)
print(f"...\n{len(merges)} слияний выучено; последние:", ", ".join(a + b for a, b in merges[-8:]))

LONG_WORDS = ["kitoblarimizdagilardan", "uylarimizda", "oʻqituvchilarimizga", "maktablarimizdagi", "talabalaringizning",
              "kutubxonalarimizdan", "qishlogʻimizga", "teragimizning", "shahrimizdagilar", "ogʻzingizdan", "burnidan",
              "bolalarimizgami"]
print()
for w in LONG_WORDS:
    gold = [x for x, _ in INVENTORY[w][1]]
    print(f"{w:<24} BPE: {' | '.join(apply_bpe(w, merges)):<36} морфемы: {' | '.join(gold)}")

In [ ]:
def boundaries(pieces):
    out, pos = set(), 0
    for p in pieces[:-1]:
        pos += len(p)
        out.add(pos)
    return out

def evaluate(segment, words):
    n_tok = tp = n_pred = n_gold = 0
    for w in words:
        pieces = segment(w)
        assert "".join(pieces) == w
        pred, gold = boundaries(pieces), boundaries([x for x, _ in INVENTORY[w][1]])
        n_tok += len(pieces); n_pred += len(pred); n_gold += len(gold); tp += len(pred & gold)
    p = tp / n_pred if n_pred else float("nan")
    r = tp / n_gold
    f = 2 * p * r / (p + r) if n_pred and p + r else 0.0
    return n_tok / len(words), p, r, f

long_nouns = sorted(w for w, (r, m) in INVENTORY.items() if r in NOUNS and len(m) >= 3)
sample = random.Random(17).sample(long_nouns, 400)
systems = {
    "пробелы": lambda w: [w],
    "символы": letters,
    "BPE-25": lambda w: apply_bpe(w, merges[:25]),
    "BPE-75": lambda w: apply_bpe(w, merges[:75]),
    f"BPE-{len(merges)}": lambda w: apply_bpe(w, merges),
    "морфемы (эталон)": lambda w: [x for x, _ in INVENTORY[w][1]],
}
res = pd.DataFrame([evaluate(f, sample) for f in systems.values()], index=list(systems),
                   columns=["фертильность", "точность", "полнота", "F1"])
print(f"{len(sample)} случайных именных форм из ≥3 морфем")
print(res.round(3).to_string(na_rep="—"), "\n")

ks = [0, 5, 10, 25, 50, 75, 100, len(merges)]
curve = np.array([evaluate(lambda w: apply_bpe(w, merges[:k]), sample) for k in ks])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(ks, curve[:, 0], "o-", color="#2B55C9")
ax[0].set_xlabel("число слияний BPE"); ax[0].set_ylabel("фертильность"); ax[0].set_title("BPE: чем больше слияний, тем меньше токенов")
for j, (lbl, c) in enumerate([("точность", "#17907E"), ("полнота", "#C94B2B"), ("F1", "#333333")]):
    ax[1].plot(ks, curve[:, j + 1], "o-", color=c, label=lbl)
ax[1].set_xlabel("число слияний BPE"); ax[1].set_ylim(0, 1.05); ax[1].set_title("Совпадение границ BPE с морфемами"); ax[1].legend()
plt.tight_layout()
plt.show()

Что видно в таблице. BPE режет длинные формы мельче, чем морфемы (около 7 токенов против 5), и даже после всех слияний примерно каждый третий его разрез проходит внутри морфемы. Частые цепочки суффиксов склеиваются в один токен (larimiz, imizning), а корни, которых в корпусе почти нет, рассыпаются на буквы (b·u·r·n·i·dan). С ростом числа слияний точность растёт, а полнота падает: BPE оптимизирует частоту пар символов, а не морфологию.

### Как это выглядит на настоящей модели

Ячейка ниже работает только в Colab (нужен интернет). Она обучает BPE библиотекой `tokenizers` на тех же предложениях и режет те же слова готовым токенизатором многоязычной модели XLM-RoBERTa (`FacebookAI/xlm-roberta-base`, скачиваются только файлы токенизатора, 10–15 МБ). Посмотрите, совпадают ли его куски с морфемами.

In [ ]:
# Только для Colab: нужен интернет. Скачиваются лишь файлы токенизатора XLM-RoBERTa (10–15 МБ, без весов модели).
try:
    from tokenizers import Tokenizer, models, trainers, pre_tokenizers
    hf_bpe = Tokenizer(models.BPE(unk_token="[UNK]", end_of_word_suffix="</w>"))
    hf_bpe.pre_tokenizer = pre_tokenizers.Whitespace()
    alphabet = {ch for w in word_counts for ch in w}
    trainer = trainers.BpeTrainer(vocab_size=len(alphabet) + len(merges) + 1, min_frequency=2,
                                  special_tokens=["[UNK]"], end_of_word_suffix="</w>", show_progress=False)
    hf_bpe.train_from_iterator([" ".join(words_of(s)) for s in UZ_CORPUS], trainer)

    from transformers import AutoTokenizer
    xlmr = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-base")
    for w in LONG_WORDS:
        print(w)
        print("   tokenizers BPE:", hf_bpe.encode(w).tokens)
        print("   XLM-R         :", xlmr.tokenize(w))
except Exception as e:
    print("Не получилось (нужен интернет и библиотеки tokenizers, transformers):", e)

## 4. Стресс-тест: наивный и устойчивый конвейер

Задача: по словоформе найти корень (лемму) из лексикона. С этого шага начинаются поиск, классификация и перевод.

- **Наивный конвейер** приводит слово к нижнему регистру, ищет его в лексиконе, а если не нашёл, отрезает один суффикс из списка.
- **Устойчивый конвейер** добавляет четыре исправления: нормализацию апострофов, транслитерацию кириллицы, разбор цепочки суффиксов по слотам и обратные правила для чередований основы (terag → terak, ogʻz → ogʻiz).

Об апострофах: по правилам в oʻ и gʻ пишется знак ʻ (U+02BB), а в тутук белгиси (разделительном знаке: maʼno «смысл») — ʼ (U+02BC). На практике люди печатают ', `, ‘, ’. Для компьютера это разные символы, а стандартная нормализация Unicode (NFKC) их не объединяет. Ниже видно и другое следствие: регулярное выражение `\w+` считает ʻ буквой, а ' нет, поэтому o'qituvchi распадается на два «слова». Узбекский текст к тому же пишут и кириллицей.

In [ ]:
import unicodedata

APOSTROPHES = "'`‘’´"            # «неправильные» апострофы; правильные ʻ и ʼ не трогаем

def norm_apostrophe(w):
    # эвристика: после o и g ставим ʻ (oʻ, gʻ), в остальных местах ʼ (тутук белгиси: maʼno, sheʼr)
    return re.sub("([ogOG])[" + APOSTROPHES + "]|[" + APOSTROPHES + "]",
                  lambda m: m.group(1) + "ʻ" if m.group(1) else "ʼ", w)

CYR = dict(zip("абвгдёжзийклмнопрстуфхцчшъьэюяўқғҳ",
               ["a", "b", "v", "g", "d", "yo", "j", "z", "i", "y", "k", "l", "m", "n", "o", "p", "r", "s", "t", "u",
                "f", "x", "ts", "ch", "sh", "ʼ", "", "e", "yu", "ya", "oʻ", "q", "gʻ", "h"]))

def cyr2lat(w):
    out = []
    for i, ch in enumerate(w):
        if ch == "е":                     # е в начале слова и после гласного читается ye
            out.append("ye" if i == 0 or w[i - 1] in "аеёиоуэюяў" else "e")
        else:
            out.append(CYR.get(ch, ch))
    return "".join(out)

word_re = re.compile(r"\w+")
for v in ["oʻqituvchi", "o'qituvchi", "o`qituvchi", "o‘qituvchi", "o’qituvchi", "ўқитувчи", "maʼno", "ma'no"]:
    ap = [ch for ch in v if ch in APOSTROPHES + "ʻʼ"]
    code = f"U+{ord(ap[0]):04X}" if ap else "—"
    nfkc_same = unicodedata.normalize("NFKC", v) in ("oʻqituvchi", "maʼno")
    fixed = norm_apostrophe(cyr2lat(v.lower()))
    print(f"{v:<11} {code:<7} \\w+ -> {str(word_re.findall(v)):<20} NFKC ok: {str(nfkc_same):<5}  норм.: {fixed}")

In [ ]:
def stem_table(alternations):
    # поверхностная основа -> корень; с чередованиями добавляем terag -> terak, ogʻz -> ogʻiz
    table = {r: r for r in NOUNS}
    if alternations:
        for r in NOUNS:
            table.setdefault(changed_stem(r), r)
    return table

NAIVE_SUFFIXES = sorted({"lar", "im", "ing", "i", "imiz", "ingiz", "m", "ng", "si", "miz", "ngiz",
                         "ning", "ni", "ga", "ka", "qa", "da", "dan", "dagi", "mi"}, key=len, reverse=True)
# цепочка суффиксов по слотам: число, принадлежность, падеж (или -da-gi-lar-падеж), -mi
SUFFIX_CHAIN = re.compile(r"(lar)?(imiz|ingiz|im|ing|i|miz|ngiz|m|ng|si)?(ning|ni|ga|ka|qa|dan|da(gi(lar)?(ni|ga|dan)?)?)?(mi)?")

def make_pipeline(apostrophe=False, cyrillic=False, chain=False, alternations=False):
    def lemmatize(word_):
        w = word_.lower()
        if cyrillic:
            w = cyr2lat(w)
        if apostrophe:
            w = norm_apostrophe(w)
        stems = stem_table(alternations)
        if chain:
            for cut in range(len(w), 0, -1):          # сначала самая длинная основа
                if w[:cut] in stems and SUFFIX_CHAIN.fullmatch(w[cut:]):
                    return stems[w[:cut]]
            return None
        if w in stems:
            return stems[w]
        for s in NAIVE_SUFFIXES:                      # отрезаем не больше одного суффикса
            if w.endswith(s) and w[:-len(s)] in stems:
                return stems[w[:-len(s)]]
        return None
    return lemmatize

naive = make_pipeline()
robust = make_pipeline(apostrophe=True, cyrillic=True, chain=True, alternations=True)

TESTS = [   # (вход, ожидаемый корень, категория)
    ("kitobda", "kitob", "чистая форма"), ("uylar", "uy", "чистая форма"), ("bolaga", "bola", "чистая форма"),
    ("maktabni", "maktab", "чистая форма"), ("dalada", "dala", "чистая форма"),
    ("bog'da", "bogʻ", "апострофы"), ("tog`lar", "togʻ", "апострофы"), ("qo‘lim", "qoʻl", "апострофы"),
    ("o’qituvchi", "oʻqituvchi", "апострофы"), ("ma’nosi", "maʼno", "апострофы"), ("she'r", "sheʼr", "апострофы"),
    ("o`g`il", "oʻgʻil", "апострофы"),
    ("китобда", "kitob", "кириллица"), ("уйлар", "uy", "кириллица"), ("болага", "bola", "кириллица"),
    ("ўқитувчи", "oʻqituvchi", "кириллица"), ("тоғда", "togʻ", "кириллица"),
    ("kitoblarimizdagilardan", "kitob", "цепочка суффиксов"), ("uylarimizda", "uy", "цепочка суффиксов"),
    ("bolalarimizgami", "bola", "цепочка суффиксов"), ("maktablarimizdagi", "maktab", "цепочка суффиксов"),
    ("dalalarimizdan", "dala", "цепочка суффиксов"),
    ("teragim", "terak", "k→g, q→gʻ"), ("qishlogʻi", "qishloq", "k→g, q→gʻ"), ("yuragimiz", "yurak", "k→g, q→gʻ"),
    ("qulogʻing", "quloq", "k→g, q→gʻ"), ("eshigi", "eshik", "k→g, q→gʻ"),
    ("ogʻzi", "ogʻiz", "выпадение гласного"), ("burnim", "burun", "выпадение гласного"), ("shahri", "shahar", "выпадение гласного"),
    ("oʻgʻlim", "oʻgʻil", "выпадение гласного"), ("qorni", "qorin", "выпадение гласного"),
    ("Шаҳримиздаги", "shahar", "комбинация"), ("O'G'LIMIZGA", "oʻgʻil", "комбинация"),
    ("qishlog'imizdagilar", "qishloq", "комбинация"), ("бурнидан", "burun", "комбинация"),
    ("КИТОБЛАРИМИЗДА", "kitob", "комбинация"), ("ўғлим", "oʻgʻil", "комбинация"),
    ("kompyuterda", None, "контроль"), ("olmalar", "olma", "контроль"), ("dalasi", "dala", "контроль"),
    ("bolam", "bola", "контроль"),
    ("boglarimizda", "bogʻ", "трудные"), ("ogzim", "ogʻiz", "трудные"),
    ("kitobcham", "kitob", "трудные"), ("bolajonim", "bola", "трудные"),
]

def mark(ok):
    return "✓" if ok else "✗"

rows = []
for w, gold, cat in TESTS:
    a, b = naive(w), robust(w)
    rows.append([w, gold or "—", cat, f"{a or '—'} {mark(a == gold)}", f"{b or '—'} {mark(b == gold)}"])
table = pd.DataFrame(rows, columns=["вход", "ожидается", "категория", "наивный", "устойчивый"])
print(table.to_string(index=False))
for name, f in [("наивный", naive), ("устойчивый", robust)]:
    n_ok = sum(f(w) == gold for w, gold, _ in TESTS)
    print(f"{name:<11} точность {n_ok}/{len(TESTS)} = {n_ok / len(TESTS):.1%}")

Каждый тест — пара «вход → ожидаемый корень» и категория искажения. Включим исправления по одному и посмотрим, как растёт точность по категориям. Категория «контроль» проверяет, что исправления ничего не сломали, а «трудные» показывают, что устойчивый конвейер тоже не всесилен: пропущенный апостроф (boglarimizda) и словообразовательные суффиксы -cha «маленький» и -jon «дорогой» (kitobcham «моя книжечка», bolajonim «мой дорогой ребёнок»), которых нет в его правилах.

Ручные тесты легко подогнать под систему. Поэтому добавим автоматические: возьмём все именные формы из инвентаря раздела 1, случайно заменим ʻ на другой «апостроф» и иногда напишем слово заглавными. Если устойчивый конвейер пройдёт все такие тесты, это значит лишь, что анализатор согласован с генератором: ошибку в правиле, общем для обоих, такой тест не найдёт. Поэтому нужны и ручные «трудные» случаи, и проверка на настоящих текстах.

In [ ]:
configs = {
    "наивный": make_pipeline(),
    "+ апострофы": make_pipeline(apostrophe=True),
    "+ кириллица": make_pipeline(apostrophe=True, cyrillic=True),
    "+ цепочки": make_pipeline(apostrophe=True, cyrillic=True, chain=True),
    "+ чередования": make_pipeline(apostrophe=True, cyrillic=True, chain=True, alternations=True),
}
cats = list(dict.fromkeys(cat for _, _, cat in TESTS))
abl = pd.DataFrame({name: [np.mean([f(w) == g for w, g, c in TESTS if c == cat]) for cat in cats]
                    for name, f in configs.items()}, index=cats)
abl.loc["точность"] = [np.mean([f(w) == g for w, g, _ in TESTS]) for f in configs.values()]
print((abl * 100).round(0).astype(int).to_string())

plt.figure(figsize=(7, 3))
plt.bar(list(configs), abl.loc["точность"] * 100, color=["#999999"] + ["#2B55C9"] * 4)
for i, v in enumerate(abl.loc["точность"] * 100):
    plt.text(i, v, f"{v:.0f}%", ha="center", va="bottom")
plt.ylabel("точность, %"); plt.ylim(0, 105)
plt.title("Точность на наборе тестов по мере добавления исправлений")
plt.tight_layout()
plt.show()

In [ ]:
rnd = random.Random(17)

def perturb(w):
    # каждую ʻ заменяем случайным «апострофом», иногда пишем слово заглавными
    w = re.sub("ʻ", lambda m: rnd.choice(["'", "`", "‘", "’"]), w)
    return w.upper() if rnd.random() < 0.3 else w

noun_forms = [(w, r) for w, (r, m) in INVENTORY.items() if r in NOUNS]
perturbed = [(perturb(w), r) for w, r in noun_forms]
print(f"Автотест на всех именных формах инвентаря: {len(noun_forms)}")
for name, f in [("наивный", naive), ("устойчивый", robust)]:
    clean_acc = np.mean([f(w) == r for w, r in noun_forms])
    pert_acc = np.mean([f(w) == r for w, r in perturbed])
    print(f"   {name:<11} без искажений: {clean_acc:6.1%}   с искажениями: {pert_acc:6.1%}")
print("  ", ", ".join(w for w, _ in perturbed[:: len(perturbed) // 8][:8]))

**А как с этим у русского?** У русского есть хороший словарный анализатор `pymorphy3`. Регистр и ё/е он переносит, но если в слове одна буква латинская (а «о», «а», «с» выглядят одинаково), он не узнаёт слово. Это тот же класс ошибок, что и узбекские апострофы, и лечится тоже нормализацией.

In [ ]:
LAT2CYR = str.maketrans("aeopcxyAEOPCXHKMTB", "аеорсхуАЕОРСХНКМТВ")

def fix_mixed(w):
    # латинские «двойники» заменяем, только если в слове смешаны алфавиты
    if re.search("[а-яё]", w, re.I) and re.search("[a-z]", w, re.I):
        return w.translate(LAT2CYR)
    return w

ru_tests = [("книгами", "книга"), ("КНИГАМИ", "книга"), ("ежиками", "ёжик"),
            ("книг" + "a" + "ми", "книга"), ("д" + "o" + "мами", "дом"), ("C" + "толами", "стол")]
for w, gold in ru_tests:
    a = morph.parse(w)[0].normal_form
    b = morph.parse(fix_mixed(w))[0].normal_form
    latin = "".join(ch for ch in w if "a" <= ch.lower() <= "z")
    print(f"{w:<9} латиница: {latin or '—':<3} pymorphy3: {a:<9} {mark(a == gold)}   после fix_mixed: {b:<6} {mark(b == gold)}")

## 5. Задания

1. **Новые тесты: пропущенный апостроф.** В переписке часто пишут ogzi, boglar, togda вместо ogʻzi, bogʻlar, togʻda. Два таких случая уже есть среди «трудных»; добавьте в `TESTS` ещё 4 с категорией «без апострофа» и пересчитайте таблицу. Затем добавьте в устойчивый конвейер запасной шаг: если слово не разобрано, попробуйте вставить ʻ после каждой o или g. Сколько тестов проходит теперь? Не сломался ли «контроль»?
2. **Новые корни.** Добавьте в `NOUNS` корни oyoq «нога» (чередование q→gʻ), oʻrin «место» и singil «младшая сестра» (выпадение гласного). Сгенерируйте все их формы через `noun_paradigm` и проверьте, что `robust` возвращает правильный корень для каждой формы. Затем запустите ячейку самопроверки.
3. **Закон Хипса.** Повторите раздел 2 для «узбекского» корпуса с 50 и с 2000 форм на лемму (для 2000 форм код словоформы `lemmas * 1000 + forms` придётся поменять: почему?). Как меняются $\beta$ и OOV при 50 000 обучающих токенов?
4. **BPE на глаголах.** Посчитайте фертильность и F1 границ BPE на глагольных формах из раздела 1 (`verb_paradigm`). Какие суффиксы BPE выучил лучше: именные или глагольные? Почему? Подсказка: посмотрите, какие формы чаще встречаются в обучающих предложениях.
5. **Автотест на кириллице.** Напишите `lat2cyr` (обратная транслитерация: oʻ → ў, gʻ → ғ, sh → ш, ch → ч, q → қ, h → ҳ, x → х, ya → я…) и прогоните автотест раздела 4 на кириллических версиях всех именных форм. Какие ошибки транслитерации он выявил?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# самопроверка к заданию 2: запустите после решения
for w, gold in [("oyogʻimizga", "oyoq"), ("oʻrnimiz", "oʻrin"), ("singlimga", "singil"), ("oyoqlarimdan", "oyoq")]:
    got = robust(w)
    print(mark(got == gold), w, "->", got, "(ожидалось:", gold + ")")